# Setup
In colab:

Go to "Runtime" -> "Change runtime type" -> Select "T4 GPU"
Install TerraTorch

In [1]:
%%time
# ── 1. Python deps ────────────────────────────────────────────────────────────
# terratorch 1.0.1 is what the notebooks were written against
# git-lfs is needed because the GeoTIFFs in examples/ are stored with LFS
!pip install --upgrade pip
!pip install terratorch==1.0.1 gdown git+https://github.com/huggingface/huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 17.6 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Cloning https://github.com/huggingface/huggingface_hub to /tmp/pip-req-build-g1xoiakz
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/huggingface_hub /tmp/pip-req-build-g1xoiakz
  Resolved https://github.com/huggingface/huggingface_hub to commit 9e0493cfdb4de5a27b45c53c3342c83ab1a138fb
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 850.8/850.8 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.9/818.9 kB 31.5 MB/s eta 0:00:

In [2]:
%%time
# ── 2. Pull the repo (only the examples/ folder) ──────────────────────────────
# --depth=1 keeps the clone quick;
# --branch selects the colab_compat branch
!git lfs install
!git clone --depth 1 --filter=blob:none --sparse \
      --branch colab_compat https://github.com/EugeneGene/terramind.git
# check out just the examples/ subtree  only
# cd back to root
%cd terramind
!git sparse-checkout set examples
%cd ..

Git LFS initialized.
Cloning into 'terramind'...
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 15 (delta 0), reused 13 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (15/15), done.
remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 0), reused 3 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (4/4), 11.65 KiB | 11.65 MiB/s, done.
/content/terramind
remote: Enumerating objects: 35, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 35 (delta 1), reused 35 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (35/35), 8.03 MiB | 20.36 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content
CPU times: user 28.8 ms, sys: 12.4 ms, total: 41.2 ms
Wall time: 1.82 s


In [4]:
%%time
# ── 3. Utility script for visualising tiles ───────────────────────────────────
!wget -q https://raw.githubusercontent.com/EugeneGene/terramind/colab_compat/notebooks/plotting_utils.py

CPU times: user 4.43 ms, sys: 797 µs, total: 5.22 ms
Wall time: 104 ms


In [5]:
%%time
# ── 4. Import Libraries relevant to this notebook ─────────────────────────────
import os
import torch
import gc
import numpy as np
import rioxarray as rxr
import matplotlib.pyplot as plt
from terratorch import FULL_MODEL_REGISTRY
# from terratorch.registry import FULL_MODEL_REGISTRY
from huggingface_hub import hf_hub_download
from plotting_utils import plot_s2, plot_modality
from terratorch.tasks.tiled_inference import tiled_inference

CPU times: user 30.7 s, sys: 5.87 s, total: 36.5 s
Wall time: 48.2 s


In [6]:
%%time
# ── 5. Select device ──────────────────────────────────────────────────────────
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
device

CPU times: user 610 µs, sys: 0 ns, total: 610 µs
Wall time: 618 µs


'cpu'

# Any-to-Any Generation

TerraMind is the first Geospatial Foundation Model (GeoFM/GFM) built with the ability to generate any modality from any other, such as predicting cloud-penetrating Sentinel-1 radar from Sentinel-2 optical, or inferring land cover maps directly from raw satellite imagery. This multimodal versatility is enabled by its architecture and training setup, which learns a shared latent space across all supported input-output combinations.
This example demonstrates how to perform flexible cross-modality generation using the generate_any_to_any function from TerraTorch.

In [7]:
%%time
# Load input data
examples = [
    '38D_378R_2_3.tif',
    '282D_485L_3_3.tif',
    '433D_629L_3_1.tif',
    '637U_59R_1_3.tif',
    '609U_541L_3_0.tif',
]

# Select example between 0 and 4
file = examples[0]

# Define modalities
modalities = ['S2L2A', 'S1RTC', 'DEM', 'LULC', 'NDVI']
data = {m: rxr.open_rasterio(f'terramind/examples/{m}/{file}') for m in modalities}
# Tensor with shape [B, C, 224, 224]
data = {
    k: torch.Tensor(v.values, device='cpu').unsqueeze(0)
    for k, v in data.items()
}

CPU times: user 75.7 ms, sys: 16.1 ms, total: 91.8 ms
Wall time: 326 ms


In [10]:
%%time
outputs = {}

for m in modalities:
    print(f'Processing {m}')
    out_modalities = modalities[:]
    out_modalities.remove(m)

    # Initialize model
    model = FULL_MODEL_REGISTRY.build(
        'terramind_v1_base_generate',
        modalities=[m],
        output_modalities=out_modalities,
        pretrained=True,
        standardize=True,
    )
    model = model.to(device)

    input_tensor = data[m].clone().to(device)

    with torch.no_grad():
        generated = model(input_tensor, verbose=True, timesteps=10)

    # Move generated outputs to CPU and detach
    outputs[m] = {
        out_m: out.detach().cpu() for out_m, out in generated.items()
    }

    # Clear memory
    del model
    del input_tensor
    torch.cuda.empty_cache()  # Only does something if you're on GPU
    gc.collect()


Processing S2L2A


TerraMind_Tokenizer_S1RTC.pt:   0%|          | 0.00/1.15G [00:00<?, ?B/s]

TerraMind_Tokenizer_DEM.pt:   0%|          | 0.00/1.15G [00:00<?, ?B/s]

TerraMind_Tokenizer_LULC.pt:   0%|          | 0.00/736M [00:00<?, ?B/s]

TerraMind_Tokenizer_NDVI.pt:   0%|          | 0.00/1.15G [00:00<?, ?B/s]

...

Processing LULC


100%|██████████| 10/10 [00:00<00:00, 13.26it/s]


Processing NDVI


100%|██████████| 10/10 [00:00<00:00, 16.44it/s]


CPU times: user 2min 28s, sys: 29.7 s, total: 2min 58s
Wall time: 5min 44s


In [11]:
%%time
# Plot any-to-any generations
n_mod = len(modalities)
fig, axes = plt.subplots(nrows=n_mod, ncols=n_mod + 1, figsize=[10, 8])

# Set titles for the top row
axes[0][0].set_title('Input')
for i, m in enumerate(modalities):
    axes[0][i + 1].set_title(m)

# Plot inputs
for (m, input), ax in zip(data.items(), axes):
    plot_modality(m, input, ax=ax[0])  # first column is the input
    for a in ax:
        a.axis('off')

# Plot generated outputs
for k, m_output in enumerate(outputs.values()):
    for m, out in m_output.items():
        j = modalities.index(m) + 1  # output columns start at index 1
        plot_modality(m, out, ax=axes[k][j])

# Save as PNG to avoid PDF memory spike
png_filename = f'any_to_any_{os.path.basename(file)}.png'
plt.tight_layout()
plt.savefig(png_filename, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved figure as {png_filename}")
plt.close(fig)  # Clear from memory to avoid crash

CPU times: user 1.46 s, sys: 25.8 ms, total: 1.48 s
Wall time: 1.8 s
